In [1]:
import numpy as np
import pandas as pd


In [3]:
# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
pbp_df       = pd.read_csv("../pbp.csv")
home_team_df = pd.read_csv("../home_team.csv")
away_team_df = pd.read_csv("../away_team.csv")
home_score_df = pd.read_csv("../home_team_score.csv")  # MatchHomeScoreInfo
away_score_df = pd.read_csv("../away_team_score.csv")  # MatchAwayScoreInfo


In [4]:

# ─────────────────────────────────────────────
# 2. CLEAN PBP DATA
# ─────────────────────────────────────────────

pbp_df = pbp_df.dropna(subset=["match_id", "set_id", "game_id"])

pbp_df["set_id"]   = pd.to_numeric(pbp_df["set_id"],  errors="coerce")
pbp_df["game_id"]  = pd.to_numeric(pbp_df["game_id"], errors="coerce")
pbp_df["match_id"] = pbp_df["match_id"].astype(int)

pbp_df = pbp_df.dropna(subset=["set_id", "game_id"])
pbp_df["set_id"]  = pbp_df["set_id"].astype(int)
pbp_df["game_id"] = pbp_df["game_id"].astype(int)

pbp_df = pbp_df[pbp_df["set_id"].between(1, 5)]
pbp_df = pbp_df[pbp_df["game_id"] > 0]
pbp_df = pbp_df.drop_duplicates(subset=["match_id", "set_id", "game_id", "point_id"])


In [5]:

# ─────────────────────────────────────────────
# 3. BUILD VALID SET SCORELINES FROM SCORE INFO
# ─────────────────────────────────────────────
# Melt home and away score tables from wide (period_1..5) → long format
# so we get the actual games won per player per set

home_sets = (
    home_score_df
    .melt(
        id_vars="match_id",
        value_vars=["period_1", "period_2", "period_3", "period_4", "period_5"],
        var_name="period_col",
        value_name="home_games_won",
    )
    .assign(set_id=lambda d: d["period_col"].str.extract(r"(\d)").astype(int))
    .drop(columns="period_col")
    .dropna(subset=["home_games_won"])
    .assign(home_games_won=lambda d: pd.to_numeric(d["home_games_won"], errors="coerce"))
    .dropna(subset=["home_games_won"])
    .assign(home_games_won=lambda d: d["home_games_won"].astype(int))
)

away_sets = (
    away_score_df
    .melt(
        id_vars="match_id",
        value_vars=["period_1", "period_2", "period_3", "period_4", "period_5"],
        var_name="period_col",
        value_name="away_games_won",
    )
    .assign(set_id=lambda d: d["period_col"].str.extract(r"(\d)").astype(int))
    .drop(columns="period_col")
    .dropna(subset=["away_games_won"])
    .assign(away_games_won=lambda d: pd.to_numeric(d["away_games_won"], errors="coerce"))
    .dropna(subset=["away_games_won"])
    .assign(away_games_won=lambda d: d["away_games_won"].astype(int))
)

# Merge home & away scores into one set-level scoreline table
set_scores = home_sets.merge(away_sets, on=["match_id", "set_id"], how="inner")

# Total games played in the set = home + away games won
set_scores["total_games_in_set"] = set_scores["home_games_won"] + set_scores["away_games_won"]


In [6]:


# ─────────────────────────────────────────────
# 4. VALIDATE LEGITIMATE SET SCORELINES
# ─────────────────────────────────────────────
# In real tennis, valid completed set scores are:
#   6-0, 6-1, 6-2, 6-3, 6-4  → total games: 6–10
#   7-5                        → total games: 12
#   7-6 (tiebreak)             → total games: 13
#   1-0 (match tiebreak)       → total games:  1  ← only allowed in final set
#
# We define a boolean mask for each rule:

h = set_scores["home_games_won"]
a = set_scores["away_games_won"]

# Standard set: one player wins 6, other wins 0–4
standard_set = (
    ((h == 6) & a.between(0, 4)) |
    ((a == 6) & h.between(0, 4))
)

# 7-5 set
set_7_5 = (
    ((h == 7) & (a == 5)) |
    ((a == 7) & (h == 5))
)

# 7-6 tiebreak set
set_7_6 = (
    ((h == 7) & (a == 6)) |
    ((a == 7) & (h == 6))
)

# Match tiebreak: scored 1-0, only valid in the final set of a match
# We identify the final set per match first
max_set_per_match = (
    set_scores
    .groupby("match_id")["set_id"]
    .max()
    .reset_index()
    .rename(columns={"set_id": "max_set_id"})
)
set_scores = set_scores.merge(max_set_per_match, on="match_id", how="left")
set_scores["is_final_set"] = set_scores["set_id"] == set_scores["max_set_id"]

match_tiebreak = (
    set_scores["is_final_set"] &
    (
        ((h == 1) & (a == 0)) |
        ((a == 1) & (h == 0))
    )
)

# Keep only sets matching one of the four legitimate patterns
valid_mask = standard_set | set_7_5 | set_7_6 | match_tiebreak
set_scores_valid = set_scores[valid_mask].copy()

print(f"Sets before score validation : {len(set_scores):,}")
print(f"Sets after  score validation : {len(set_scores_valid):,}")
print(f"Sets dropped (invalid scores): {(~valid_mask).sum():,}\n")


Sets before score validation : 151,705
Sets after  score validation : 133,222
Sets dropped (invalid scores): 18,483



In [7]:

# ─────────────────────────────────────────────
# 5. COUNT GAMES PER SET FROM PBP & CROSS-JOIN
# ─────────────────────────────────────────────
# Count distinct game_ids within each (match_id, set_id) from the pbp data
games_per_set_pbp = (
    pbp_df
    .groupby(["match_id", "set_id"])["game_id"]
    .nunique()
    .reset_index()
    .rename(columns={"game_id": "num_games"})
)

# Inner join: only keep (match, set) pairs that are present in BOTH
# the pbp data AND the validated score table — dual-source quality gate
analysis_df_sets = games_per_set_pbp.merge(
    set_scores_valid[["match_id", "set_id", "total_games_in_set"]],
    on=["match_id", "set_id"],
    how="inner",
)

# Use total_games_in_set from the official score as the ground-truth game count
# (more reliable than counting game_ids in pbp, which can have missing points)
analysis_df_sets = analysis_df_sets.rename(
    columns={"total_games_in_set": "num_games_official"}
)


In [8]:

# ─────────────────────────────────────────────
# 6. CLEAN TEAM INFO & EXTRACT GENDER
# ─────────────────────────────────────────────
home_gender = (
    home_team_df[["match_id", "gender"]]
    .dropna(subset=["gender"])
    .assign(gender=lambda d: d["gender"].str.strip().str.upper())
    .rename(columns={"gender": "home_gender"})
)
away_gender = (
    away_team_df[["match_id", "gender"]]
    .dropna(subset=["gender"])
    .assign(gender=lambda d: d["gender"].str.strip().str.upper())
    .rename(columns={"gender": "away_gender"})
)

gender_df = (
    home_gender
    .merge(away_gender, on="match_id", how="inner")
    .query("home_gender == away_gender and home_gender in ['M', 'F']")
    .assign(gender=lambda d: d["home_gender"])
    [["match_id", "gender"]]
    .drop_duplicates()
)


In [9]:

# ─────────────────────────────────────────────
# 7. JOIN & FINAL SANITY CHECK
# ─────────────────────────────────────────────
analysis_df = analysis_df_sets.merge(gender_df, on="match_id", how="inner")

assert set(analysis_df["gender"].unique()).issubset({"M", "F"}), \
    "Unexpected gender values found after filtering"

# Confirm no impossible game counts survive
min_games = analysis_df["num_games_official"].min()
assert min_games >= 1, f"Unexpected min games: {min_games}"
# Match tiebreaks (1 game) are now the only legitimate 1-game sets
print(f"Total (match, set) records after full cleaning : {len(analysis_df):,}")
print(f"Unique matches                                 : {analysis_df['match_id'].nunique():,}")
print(f"Min games in a set                             : {min_games}")
print(f"Gender distribution:\n{analysis_df['gender'].value_counts()}\n")


Total (match, set) records after full cleaning : 79,734
Unique matches                                 : 8,579
Min games in a set                             : 1
Gender distribution:
gender
M    45197
F    34537
Name: count, dtype: int64



In [10]:

# ─────────────────────────────────────────────
# 8. COMPUTE AVERAGE GAMES PER SET BY GENDER
# ─────────────────────────────────────────────

# 8a. Exclude match tiebreaks (1-game sets) from the average —
#     they are structurally different from normal sets and skew the mean
analysis_normal = analysis_df[analysis_df["num_games_official"] > 1].copy()

result = (
    analysis_normal
    .groupby("gender")["num_games_official"]
    .agg(
        max_games="max",
        min_games="min",
        avg_games_per_set="mean",
        median_games_per_set="median",
        std_games_per_set="std",
        total_sets="count",
    )
    .reset_index()
    .assign(
        avg_games_per_set=lambda d: d["avg_games_per_set"].round(3),
        median_games_per_set=lambda d: d["median_games_per_set"].round(3),
        std_games_per_set=lambda d: d["std_games_per_set"].round(3),
        gender=lambda d: d["gender"].map({"M": "Men", "F": "Women"}),
    )
)

print("=" * 55)
print("  Average Number of Games per Set: Men vs Women")
print("  (Match tiebreaks excluded)")
print("=" * 55)
print(result.to_string(index=False))
print()


  Average Number of Games per Set: Men vs Women
  (Match tiebreaks excluded)
gender  max_games  min_games  avg_games_per_set  median_games_per_set  std_games_per_set  total_sets
 Women         13          6              9.180                   9.0              1.957       34487
   Men         13          6              9.549                   9.0              1.970       45161



In [11]:

# ─────────────────────────────────────────────
# 9. BREAKDOWN BY SET NUMBER
# ─────────────────────────────────────────────
set_breakdown = (
    analysis_normal
    .groupby(["gender", "set_id"])["num_games_official"]
    .agg(avg_games="mean", total_sets="count")
    .round(3)
    .reset_index()
    .assign(gender=lambda d: d["gender"].map({"M": "Men", "F": "Women"}))
    .rename(columns={"set_id": "Set", "gender": "Gender",
                     "avg_games": "Avg Games", "total_sets": "Total Sets"})
)

print("=" * 55)
print("  Average Games per Set — by Set Number")
print("  (Match tiebreaks excluded)")
print("=" * 55)
print(set_breakdown.to_string(index=False))

  Average Games per Set — by Set Number
  (Match tiebreaks excluded)
Gender  Set  Avg Games  Total Sets
 Women    1      9.196       15189
 Women    2      9.166       14867
 Women    3      9.171        4431
   Men    1      9.610       19747
   Men    2      9.489       19442
   Men    3      9.541        5972
